# Importing Packages

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, when, to_date, trim, to_timestamp, regexp_replace, lag, sum, date_trunc
from pyspark.sql.window import Window
from pyspark.sql.types import *
load_dotenv()

# Defining Variables

In [0]:
SILVER_SCHEMA_PATH=os.getenv('SILVER_SCHEMA_PATH')
BRONZE_SCHEMA_PATH=os.getenv('BRONZE_SCHEMA_PATH')

# Date Functions

In [0]:
def parse_mixed_date(df, col_name, output_col=None):
    out_col = output_col if output_col else col_name

    return df.withColumn(
        out_col,
        when(
            trim(col(col_name)).rlike(r"^\d{1,2}-\d{1,2}-\d{4}$"),
            to_date(trim(col(col_name)), "dd-MM-yyyy")
        ).when(
            trim(col(col_name)).rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"),
            to_date(trim(col(col_name)),"dd/MM/yyyy")
        )
        .when(
            trim(col(col_name)).rlike(r"^\d{4}/\d{1,2}/\d{1,2}$"),
            to_date(trim(col(col_name)), "yyyy/MM/dd")
        )
        .when(
            trim(col(col_name)).rlike(r"^\d{4}-\d{1,2}-\d{1,2}$"),
            to_date(trim(col(col_name)), "yyyy-MM-dd")
        ).otherwise(None)
    )

# Date Cleaning

## Product Table

In [0]:
product_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_product`""")

### Trimming Spaces

In [0]:
product_df=product_df.withColumn("product_name",trim(col("product_name")))

### Typecasting

In [0]:
product_df = product_df.\
            withColumn("is_active",
            when(col("is_active")=="Y",True)
            .when(col("is_active")=="N",False)
            .otherwise(None)
            )

In [0]:
product_df=parse_mixed_date(product_df,"created_date")

In [0]:
product_df=product_df.withColumns(
    {
        "list_price":col("list_price").cast(IntegerType()),
        "product_id":col("product_id").cast(IntegerType())
    }
    )

### Creating Surrogate Key

In [0]:
product_df.createOrReplaceTempView("prod_temp")
product_key=spark.sql("""
with cte as (
  select distinct product_name,plan_name,billing_cycle,created_date from prod_temp
)
select product_name,plan_name,billing_cycle,created_date,row_number() over(order by product_name) as product_sk from cte 
""")

In [0]:
product_df=product_df.join(product_key,["product_name","plan_name","billing_cycle","created_date"],"left")

# Creating Schema

In [0]:
spark.sql(f"""create schema if not exists {SILVER_SCHEMA_PATH}""")

# Saving Dataframe

## Applying SCD Type 1 for Products Table

In [0]:
from delta.tables import DeltaTable

target_table_name = f"{SILVER_SCHEMA_PATH}.silver_product"
if spark.catalog.tableExists(target_table_name):
    deltaTable = DeltaTable.forName(spark, target_table_name)
    
    deltaTable.alias("target").merge(
        product_df.alias("updates"),
        "target.product_id = updates.product_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    product_df.write.format("delta").mode("overwrite").saveAsTable(target_table_name)